# TBI Incident — Machine-Learning Experiments

Predictive modelling on the TBI incident cohort. Every experiment is run **twice** — once with
**main-category** incident features and once with **subcategory** incident features.

## Table of contents

1. **Setup & preprocessing** — load data, classify ICD external-cause codes, derive `had_icu`,
   `n_incident_cats`, `discharge_group`, build the feature/modeling helpers.
2. **Experiment 1 — Predicting ICU requirement** (binary classification): Logistic Regression,
   Random Forest, XGBoost.
3. **Experiment 2 — Predicting discharge destination** (multiclass): Random Forest, LightGBM, XGBoost.
4. **Experiment 3 — Predicting length of stay** (regression): Negative Binomial, Poisson,
   Random Forest, XGBoost — for Hospital LOS (3A/3B) and ICU LOS (3C/3D).
5. **Experiment 4 — Demographics-only prediction** (age + gender + incident → outcomes):
   Linear/Logistic Regression, Random Forest, XGBoost, CatBoost — 4A–4F.

**Data notes.** There is no `had_icu` column; it is derived as `icu_los_days.notna()` (→ 1/0).
Incident classification reuses the CDC/NCHS external-cause ICD mapping from the EDA notebook.
`discharge_location` is collapsed into **7 buckets**. All splits are **80/20**
(`random_state=42`, stratified for classification).

## 1 · Setup & preprocessing

In [ ]:
# !pip install scikit-learn xgboost lightgbm catboost statsmodels seaborn openpyxl
import re, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, accuracy_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)

from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, CatBoostRegressor
import statsmodels.api as sm

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"]       = 110
plt.rcParams["axes.titlesize"]   = 12
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

# ---- consistent colours ----
MAIN_COLORS = {
    "Fall": "#4C72B0", "Transportation": "#DD8452", "Struck by/against": "#55A868",
    "Assault": "#C44E52", "Unspecified": "#8C8C8C", "Late effect": "#937860",
    "Natural/environmental": "#64B5CD", "Multiple": "#8172B3",
}
MODEL_COLORS = {
    "Logistic Regression": "#4C72B0", "Linear Regression": "#4C72B0",
    "Random Forest": "#55A868", "XGBoost": "#DD8452", "LightGBM": "#C44E52",
    "CatBoost": "#8172B3", "Poisson": "#937860", "Negative Binomial": "#64B5CD",
}

MAIN_GROUP_ORDER = ["Fall", "Transportation", "Struck by/against",
                    "Assault", "Unspecified", "Multiple"]
DISCH_ORDER = ["Home", "Rehab", "Skilled Nursing Facility",
               "Long-Term Care Facility", "Hospice", "Dead/Expired", "Other"]
TOP_N = {"Fall": 17, "Transportation": 23, "Struck by/against": 8,
         "Assault": 6, "Unspecified": 2}
MULTI_COMBOS = ["Fall + Struck by/against", "Fall + Unspecified",
                "Fall + Transportation", "Fall + Late effect",
                "Assault + Struck by/against"]

print("Libraries loaded.")

In [ ]:
PATIENT_CSV = "TBI_Incident_dataset.csv"
MATRIX_XLSX = "ICD_External_Cause_Matrix_CDC_Verified.xlsx"

df = pd.read_csv(PATIENT_CSV).drop_duplicates(subset="hadm_id").reset_index(drop=True)

# ---- normalise the age column name ----
df["age"] = df["age_at_admission"] if "age_at_admission" in df.columns else df.get("age")

# ============================================================
#  CDC external-cause ICD code -> (description, family)
#  (same logic as the EDA v3 notebook)
# ============================================================
DASH = "–"

def family(mech):
    """Collapse a detailed CDC mechanism into one main family, or None."""
    m = str(mech).strip()
    if "Fall" in m:
        return "Fall"
    if (m.startswith("MVT") or m == "MV" + DASH + "Nontraffic" or m.startswith("Transport")
            or m in {"Other land transport", "Other transport",
                     "Pedestrian" + DASH + "other", "Pedal cyclist" + DASH + "other"}):
        return "Transportation"
    if m.startswith("Assault"):   return "Assault"
    if "Struck" in m:             return "Struck by/against"
    if "Late effect" in m:        return "Late effect"
    if "Natural" in m:            return "Natural/environmental"
    if m == "Unspecified":        return "Unspecified"
    return None

def expand9_full(rng):
    rng = str(rng).strip()
    if rng.startswith("["):
        return []
    m = re.match(rf"E(\d{{3}})\.(\d){DASH}E(\d{{3}})\.(\d)$", rng)
    if m and m.group(1) == m.group(3):
        p = f"E{int(m.group(1)):03d}"
        return [f"{p}{d}" for d in range(int(m.group(2)), int(m.group(4)) + 1)]
    m = re.match(r"E(\d{3})\.(\d)$", rng)
    if m:
        return [f"E{int(m.group(1)):03d}{m.group(2)}"]
    m = re.match(rf"E(\d{{3}}){DASH}E(\d{{3}})$", rng)
    if m:
        out = []
        for n in range(int(m.group(1)), int(m.group(2)) + 1):
            p = f"E{n:03d}"
            out.append(p)
            out.extend(f"{p}{d}" for d in range(10))
        return out
    m = re.match(r"E(\d{3})$", rng)
    if m:
        p = f"E{int(m.group(1)):03d}"
        return [p] + [f"{p}{d}" for d in range(10)]
    return []

def expand10_full(rng):
    head = str(rng).split("(")[0].strip()
    m = re.match(rf"([VWXY])(\d{{2}}){DASH}([VWXY])(\d{{2}})$", head)
    if m and m.group(1) == m.group(3):
        return [f"{m.group(1)}{n:02d}" for n in range(int(m.group(2)), int(m.group(4)) + 1)]
    m = re.match(r"([VWXY]\d{2})(\.\d+)?$", head)
    if m:
        return [m.group(1)]
    return []

df9 = pd.read_excel(MATRIX_XLSX, sheet_name="ICD-9 E-Codes (Mechanism)", header=3)
df9.columns = ["code_range", "description", "mechanism", "intent", "notes"]
df9 = df9.dropna(subset=["code_range", "description", "mechanism"])
df10 = pd.read_excel(MATRIX_XLSX, sheet_name="ICD-10 V-Y Codes (Mechanism)", header=3)
df10.columns = ["code_range", "description", "mechanism", "intent", "notes"]
df10 = df10.dropna(subset=["code_range", "description", "mechanism"])

EXACT9 = {}; SUB9 = defaultdict(set)
for _, r in df9.iterrows():
    fam = family(r["mechanism"])
    if fam is None:
        continue
    for c in expand9_full(r["code_range"]):
        EXACT9.setdefault(c, (r["description"], fam))
        if len(c) >= 4:
            SUB9[c[:4]].add((r["description"], fam))
PREFIX9 = {p: next(iter(s)) for p, s in SUB9.items() if len({f for _, f in s}) == 1}

EXACT10 = {}
for _, r in df10.iterrows():
    fam = family(r["mechanism"])
    if fam is None:
        continue
    for c in expand10_full(r["code_range"]):
        EXACT10.setdefault(c, (r["description"], fam))

def lookup_icd9(code):
    if code in EXACT9:                          return EXACT9[code]
    if len(code) >= 5 and code[:5] in EXACT9:   return EXACT9[code[:5]]
    if len(code) >= 4 and code[:4] in EXACT9:   return EXACT9[code[:4]]
    if len(code) >= 4 and code[:4] in PREFIX9:  return PREFIX9[code[:4]]
    return None

def lookup_icd10(code):
    return EXACT10.get(code[:3])

def classify(all_codes, version):
    main = set(); subs = []
    if pd.isna(all_codes):
        return main, subs
    for raw in str(all_codes).split(","):
        c = raw.strip().upper()
        if not c:
            continue
        if int(version) == 9:
            if not c.startswith("E"):  continue
            hit = lookup_icd9(c)
        else:
            if c[0] not in "VWXY":     continue
            hit = lookup_icd10(c)
        if hit:
            desc, fam = hit
            main.add(fam); subs.append((fam, desc))
    return main, subs

_m, _s = zip(*df.apply(lambda r: classify(r["all_icd_codes"], r["icd_version"]), axis=1))
df["main_set"]        = list(_m)
df["sub_hits"]        = list(_s)
df["n_incident_cats"] = df["main_set"].apply(len)

def assign_main(s):
    if len(s) == 0:  return None
    if len(s) == 1:  return next(iter(s))
    return "Multiple"
df["main_category"] = df["main_set"].apply(assign_main)

# ---- target derivations ----
df["had_icu"] = df["icu_los_days"].notna()

def discharge_bucket(row):
    if row["hospital_expire_flag"] == 1:
        return "Dead/Expired"
    loc = row["discharge_location"]
    if pd.isna(loc):
        return np.nan
    loc = str(loc).strip().upper()
    mapping = {
        "HOME": "Home", "REHAB": "Rehab",
        "SKILLED NURSING FACILITY": "Skilled Nursing Facility",
        "CHRONIC/LONG TERM ACUTE CARE": "Long-Term Care Facility",
        "HOSPICE": "Hospice", "DIED": "Dead/Expired",
    }
    return mapping.get(loc, "Other")
df["discharge_group"] = df.apply(discharge_bucket, axis=1)

# ---- tidy demographic columns ----
df["gender"]    = df["gender"].fillna("Unknown").astype(str)
df["insurance"] = df["insurance"].fillna("Unknown").astype(str)

# ---- modeling cohort: the six main incident groups ----
cat_df  = df[df["main_category"].notna()].copy()
df_main = df[df["main_category"].isin(MAIN_GROUP_ORDER)].copy()

print(f"Admissions: {len(df):,}")
print(f"Classified (any family): {len(cat_df):,}")
print(f"Modeling cohort (6 main groups): {len(df_main):,}")
print("\nMain-category counts:")
print(df_main["main_category"].value_counts().reindex(MAIN_GROUP_ORDER).to_string())

In [ ]:
# ---- choose the top-N subcategory descriptions per family ----
def patient_sub_counts(main_cat):
    counts = Counter()
    for subs in cat_df["sub_hits"]:
        seen = set()
        for fam, desc in subs:
            if fam == main_cat and desc not in seen:
                counts[desc] += 1; seen.add(desc)
    return pd.Series(counts).sort_values(ascending=False)

SUB_LABELS = {fam: list(patient_sub_counts(fam).head(n).index) for fam, n in TOP_N.items()}

print("Subcategories kept per family:")
for fam, labs in SUB_LABELS.items():
    print(f"  {fam:<20}: {len(labs)}")
print("\nMultiple combos used as subgroup features:", len(MULTI_COMBOS))

### Discharge-bucket mapping

| Bucket | Original `discharge_location` (or flag) |
|--------|------------------------------------------|
| **Dead/Expired** | `hospital_expire_flag == 1`, `DIED` |
| **Home** | `HOME` |
| **Rehab** | `REHAB` |
| **Skilled Nursing Facility** | `SKILLED NURSING FACILITY` |
| **Long-Term Care Facility** | `CHRONIC/LONG TERM ACUTE CARE` |
| **Hospice** | `HOSPICE` |
| **Other** | `HOME HEALTH CARE`, `AGAINST ADVICE`, `OTHER FACILITY`, `PSYCH FACILITY`, `ACUTE HOSPITAL`, `ASSISTED LIVING`, `HEALTHCARE FACILITY` |

Rows with a **missing** `discharge_location` (and not flagged as expired) are excluded from
discharge-destination targets. The cell below confirms exactly what landed in **Other**.

In [ ]:
other_raw = (df.loc[df["discharge_group"] == "Other", "discharge_location"]
               .fillna("(missing)").value_counts())
print("Raw discharge_location values mapped to 'Other':")
print(other_raw.to_string())
print("\nDischarge-group counts (modeling cohort):")
print(df_main["discharge_group"].value_counts(dropna=False).reindex(DISCH_ORDER).to_string())

### Feature engineering & modeling helpers

**Incident features.**
- *Category level* — one-hot of `main_category` (6 mutually-exclusive groups, `drop_first` to avoid
  the dummy trap).
- *Subcategory level* — **multi-hot** indicator columns: one per shortlisted subcategory
  (Fall 17 + Transportation 23 + Struck 8 + Assault 6 + Unspecified 2 = 56) plus 5 **Multiple-combo**
  indicators. A patient with several subcodes in one family lights up several indicators at once, so
  the multi-category structure is preserved without multi-counting patients (one row per patient).

`n_incident_cats` carries how many distinct families a patient matched. Numeric NaNs are
median-imputed; `gender`/`insurance` use one-hot encoding.

The helpers below standardise the train/test/evaluate loop. `GLMCount` wraps statsmodels Poisson /
Negative-Binomial GLMs behind a scikit-learn-style `fit`/`predict` interface (dropping zero-variance
columns and adding an intercept) so all four LOS models share one code path.

In [ ]:
# ---------- feature builders ----------
def build_cat_features(frame):
    return pd.get_dummies(frame["main_category"], prefix="cat", drop_first=True).astype(int)

def build_sub_features(frame):
    cols = {}
    for fam in ["Fall", "Transportation", "Struck by/against", "Assault", "Unspecified"]:
        for desc in SUB_LABELS[fam]:
            cols[f"sub::{fam}::{desc}"] = frame["sub_hits"].apply(
                lambda hs, f=fam, d=desc: int(any(a == f and b == d for a, b in hs)))
    for combo in MULTI_COMBOS:
        cset = frozenset(combo.split(" + "))
        cols[f"combo::{combo}"] = frame["main_set"].apply(
            lambda s, cs=cset: int(frozenset(s) == cs))
    return pd.DataFrame(cols, index=frame.index).astype(int)

def build_extra(frame, include):
    parts = []
    if "n_incident_cats" in include:   parts.append(frame[["n_incident_cats"]].astype(float))
    if "age" in include:               parts.append(frame[["age"]].astype(float))
    if "last_gcs_score" in include and "last_gcs_score" in frame.columns:
        parts.append(frame[["last_gcs_score"]].astype(float))
    if "gender" in include:            parts.append(pd.get_dummies(frame["gender"], prefix="gender", drop_first=True).astype(int))
    if "insurance" in include:         parts.append(pd.get_dummies(frame["insurance"], prefix="ins", drop_first=True).astype(int))
    if "had_icu" in include:           parts.append(frame[["had_icu"]].astype(int))
    if "hospital_los_days" in include: parts.append(frame[["hospital_los_days"]].astype(float))
    return pd.concat(parts, axis=1) if parts else pd.DataFrame(index=frame.index)

def make_X(frame, level, include):
    inc = build_cat_features(frame) if level == "category" else build_sub_features(frame)
    X = pd.concat([inc, build_extra(frame, include)], axis=1)
    X = X.fillna(X.median(numeric_only=True)).fillna(0)
    return X

# ---------- statsmodels count-model wrapper ----------
class GLMCount:
    def __init__(self, kind):  self.kind = kind
    def fit(self, X, y):
        Xv = X.loc[:, X.nunique() > 1].astype(float)
        self.cols = list(Xv.columns)
        Xc = sm.add_constant(Xv, has_constant="add")
        self.design = list(Xc.columns)
        fam = sm.families.Poisson() if self.kind == "poisson" \
              else sm.families.NegativeBinomial(alpha=1.0)
        self.res = sm.GLM(np.asarray(y, float), Xc, family=fam).fit(maxiter=100)
        return self
    def predict(self, X):
        Xv = X.reindex(columns=self.cols, fill_value=0).astype(float)
        Xc = sm.add_constant(Xv, has_constant="add").reindex(columns=self.design, fill_value=0)
        return np.asarray(self.res.predict(Xc))

# ---------- comparison bar chart (one subplot per metric) ----------
def comparison_chart(metrics_df, metrics, title):
    n = len(metrics)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, m in zip(axes, metrics):
        vals = metrics_df[m]
        ax.bar(vals.index, vals.values,
               color=[MODEL_COLORS.get(x, "#888888") for x in vals.index])
        ax.set_title(m); ax.tick_params(axis="x", rotation=25)
        for i, v in enumerate(vals.values):
            if pd.notna(v):
                ax.annotate(f"{v:.3f}", (i, v), ha="center", va="bottom", fontsize=8)
    plt.suptitle(title, y=1.03); plt.tight_layout(); plt.show()

# ---------- classification runner ----------
def run_classification(X, y, model_specs, title, plot_roc=False):
    le = LabelEncoder()
    yenc = le.fit_transform(y)
    classes = le.classes_
    binary = len(classes) == 2
    Xtr, Xte, ytr, yte = train_test_split(X, yenc, test_size=0.2,
                                          random_state=RANDOM_STATE, stratify=yenc)
    names = list(model_specs)
    results = {}; roc_data = {}
    fig, axc = plt.subplots(1, len(names), figsize=(4.6 * len(names), 4))
    if len(names) == 1:
        axc = [axc]
    for ax, name in zip(axc, names):
        model = model_specs[name]
        model.fit(Xtr, ytr)
        pred = np.asarray(model.predict(Xte)).reshape(-1)
        if pred.dtype.kind == "f":
            pred = pred.round().astype(int)
        proba = model.predict_proba(Xte) if hasattr(model, "predict_proba") else None
        acc = accuracy_score(yte, pred)
        f1w = f1_score(yte, pred, average="weighted")
        f1m = f1_score(yte, pred, average="macro")
        auc = np.nan
        try:
            if binary and proba is not None:
                auc = roc_auc_score(yte, proba[:, 1])
            elif proba is not None:
                auc = roc_auc_score(yte, proba, multi_class="ovr", average="weighted")
        except Exception:
            pass
        results[name] = dict(Accuracy=acc, F1_weighted=f1w, F1_macro=f1m, ROC_AUC=auc)
        print(f"\n----- {name}  |  {title} -----")
        print(classification_report(yte, pred, target_names=[str(c) for c in classes]))
        cm = confusion_matrix(yte, pred)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                    xticklabels=[str(c) for c in classes],
                    yticklabels=[str(c) for c in classes])
        ax.set_title(f"{name}\nConfusion matrix")
        ax.set_xlabel("Predicted"); ax.set_ylabel("True")
        ax.tick_params(axis="x", rotation=45); ax.tick_params(axis="y", rotation=0)
        if plot_roc and binary and proba is not None:
            roc_data[name] = (yte, proba[:, 1])
    plt.suptitle(title, y=1.04); plt.tight_layout(); plt.show()
    if plot_roc and roc_data:
        fig, ax = plt.subplots(figsize=(6, 5))
        for name, (yt, pp) in roc_data.items():
            fpr, tpr, _ = roc_curve(yt, pp)
            ax.plot(fpr, tpr, label=f"{name} (AUC={results[name]['ROC_AUC']:.3f})",
                    color=MODEL_COLORS.get(name, "#888888"))
        ax.plot([0, 1], [0, 1], "k--", alpha=.5)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
        ax.set_title(f"ROC curves — {title}"); ax.legend()
        plt.tight_layout(); plt.show()
    return pd.DataFrame(results).T

# ---------- regression runner ----------
def run_regression(X, y, model_specs, title):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    results = {}; preds = {}
    print(f"===== {title} =====")
    for name, est in model_specs.items():
        try:
            est.fit(Xtr, ytr)
            p = np.asarray(est.predict(Xte), dtype=float)
            results[name] = dict(MAE=mean_absolute_error(yte, p),
                                 RMSE=mean_squared_error(yte, p) ** 0.5,
                                 R2=r2_score(yte, p))
            preds[name] = p
            print(f"  {name:<20}  MAE={results[name]['MAE']:.3f}  "
                  f"RMSE={results[name]['RMSE']:.3f}  R2={results[name]['R2']:.3f}")
        except Exception as e:
            print(f"  {name:<20}  FAILED: {e}")
            results[name] = dict(MAE=np.nan, RMSE=np.nan, R2=np.nan)
    rdf = pd.DataFrame(results).T
    comparison_chart(rdf, ["MAE", "R2"], f"{title} — model comparison")
    valid = rdf.dropna(subset=["R2"])
    if not valid.empty:
        best = valid["R2"].idxmax()
        yte_arr = np.asarray(yte, dtype=float)
        fig, ax = plt.subplots(figsize=(5.5, 5.5))
        ax.scatter(yte_arr, preds[best], alpha=.3, s=12, color="#4C72B0")
        hi = max(yte_arr.max(), preds[best].max())
        ax.plot([0, hi], [0, hi], "r--")
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.set_title(f"{title}\nBest model: {best}  (R2={rdf.loc[best, 'R2']:.3f})")
        plt.tight_layout(); plt.show()
    return rdf

# ---------- logistic-regression odds ratios (statsmodels) ----------
def logistic_odds_ratios(X, y, title):
    Xv = X.loc[:, X.nunique() > 1].astype(float)
    Xc = sm.add_constant(Xv, has_constant="add")
    try:
        res = sm.Logit(np.asarray(y, int), Xc).fit(disp=0, maxiter=200)
    except Exception as e:
        print(f"Logit did not converge: {e}")
        return None
    ci = res.conf_int()
    tbl = pd.DataFrame({"OR": np.exp(res.params),
                        "CI_low": np.exp(ci[0]), "CI_high": np.exp(ci[1]),
                        "p_value": res.pvalues}).drop("const", errors="ignore")
    tbl = tbl.sort_values("OR")
    print(f"Odds ratios — {title}")
    print(tbl.round(3).to_string())
    fig, ax = plt.subplots(figsize=(8, max(4, 0.34 * len(tbl))))
    ax.errorbar(tbl["OR"], range(len(tbl)),
                xerr=[tbl["OR"] - tbl["CI_low"], tbl["CI_high"] - tbl["OR"]],
                fmt="o", color="#4C72B0", ecolor="#AAAAAA", capsize=3)
    ax.axvline(1, color="red", ls="--")
    ax.set_yticks(range(len(tbl))); ax.set_yticklabels(tbl.index, fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("Odds ratio (log scale, 95% CI)")
    ax.set_title(title)
    plt.tight_layout(); plt.show()
    return tbl

print("Helpers ready.")

## Experiment 1 · Predicting ICU Requirement (binary classification)

**Target:** `had_icu` (1 = admitted to ICU, 0 = not). **Features:** incident category/subcategory,
`n_incident_cats`, `age`, `last_gcs_score` (admission-time severity), `gender`, `insurance`.

**Models.**
- **Logistic Regression** — linear log-odds model; interpretable via **odds ratios** (OR > 1 raises
  the odds of ICU, OR < 1 lowers it).
- **Random Forest** — bagged decision trees; captures non-linearities and interactions, robust to scale.
- **XGBoost** — gradient-boosted trees; typically the strongest tabular learner.

For every model we report the **classification report** (precision/recall/F1/accuracy),
**confusion matrix**, and **ROC-AUC**, then compare the three models side by side.

### Round 1 · By main category

In [ ]:
COH1 = df_main.copy()
y1   = COH1["had_icu"].astype(int)
INC1 = ["n_incident_cats", "age", "last_gcs_score", "gender", "insurance"]
X1c  = make_X(COH1, "category", INC1)

specs = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                                         eval_metric="logloss", random_state=RANDOM_STATE),
}
res1c = run_classification(X1c, y1, specs, "Exp 1 — ICU requirement (categories)", plot_roc=True)
comparison_chart(res1c, ["Accuracy", "F1_weighted", "ROC_AUC"],
                 "Exp 1 (categories) — model comparison")
logistic_odds_ratios(X1c, y1, "Exp 1 (categories) — Logistic Regression odds ratios")

### Round 2 · By subcategory

In [ ]:
X1s = make_X(COH1, "subcategory", INC1)
specs = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.1,
                                         eval_metric="logloss", random_state=RANDOM_STATE),
}
res1s = run_classification(X1s, y1, specs, "Exp 1 — ICU requirement (subcategories)", plot_roc=True)
comparison_chart(res1s, ["Accuracy", "F1_weighted", "ROC_AUC"],
                 "Exp 1 (subcategories) — model comparison")
logistic_odds_ratios(X1s, y1, "Exp 1 (subcategories) — Logistic Regression odds ratios")

**Interpretation.** Compare ROC-AUC across models and rounds: if the subcategory round barely
beats the category round, the fine-grained mechanism adds little ICU-predictive signal beyond the broad
family plus demographics/GCS. Odds ratios show *which* incident types and patient factors raise ICU
odds — a low admission GCS, for example, should push the odds ratio well above 1.

## Experiment 2 · Predicting Discharge Destination (multiclass classification)

**Target:** `discharge_group` (7 buckets). **Features:** incident category/subcategory,
`n_incident_cats`, `age`, `gender`, `insurance`, `had_icu`, `hospital_los_days`.

**Models:** **Random Forest**, **LightGBM** (fast leaf-wise gradient boosting), **XGBoost**.
Reported per model: classification report (per-class precision/recall/F1), confusion-matrix heatmap,
and macro/weighted F1. The class mix is imbalanced (most patients go Home), so **weighted F1** reflects
overall accuracy while **macro F1** reveals performance on the rarer destinations.

### Round 1 · By main category

In [ ]:
COH2 = df_main[df_main["discharge_group"].notna()].copy()
y2   = COH2["discharge_group"]
INC2 = ["n_incident_cats", "age", "gender", "insurance", "had_icu", "hospital_los_days"]
X2c  = make_X(COH2, "category", INC2)

specs = {
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM":      LGBMClassifier(n_estimators=400, random_state=RANDOM_STATE, verbose=-1),
    "XGBoost":       XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.1,
                                   eval_metric="mlogloss", random_state=RANDOM_STATE),
}
res2c = run_classification(X2c, y2, specs, "Exp 2 — discharge destination (categories)")
comparison_chart(res2c, ["Accuracy", "F1_weighted"], "Exp 2 (categories) — model comparison")
print("Macro F1:\n", res2c["F1_macro"].round(3).to_string())

### Round 2 · By subcategory

In [ ]:
X2s = make_X(COH2, "subcategory", INC2)
specs = {
    "Random Forest": RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "LightGBM":      LGBMClassifier(n_estimators=400, random_state=RANDOM_STATE, verbose=-1),
    "XGBoost":       XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.1,
                                   eval_metric="mlogloss", random_state=RANDOM_STATE),
}
res2s = run_classification(X2s, y2, specs, "Exp 2 — discharge destination (subcategories)")
comparison_chart(res2s, ["Accuracy", "F1_weighted"], "Exp 2 (subcategories) — model comparison")
print("Macro F1:\n", res2s["F1_macro"].round(3).to_string())

**Interpretation.** `hospital_los_days` and `had_icu` are usually the dominant predictors here —
longer, ICU-involving stays shift discharge away from Home toward rehab / skilled nursing / long-term
care. Incident mechanism contributes secondary signal. The confusion-matrix heatmaps show which
destinations get confused (e.g. Skilled Nursing vs. Long-Term Care).

## Experiment 3 · Predicting Length of Stay (regression)

**Targets:** `hospital_los_days` (3A/3B) and `icu_los_days` for ICU patients (3C/3D).
**Features:** incident category/subcategory, `n_incident_cats`, `age`, `gender`, `insurance`, `had_icu`.
**Models:** **Negative Binomial**, **Poisson**, **Random Forest**, **XGBoost**.

**Why count models instead of ordinary linear regression?** Length of stay is a **non-negative count
of days** with a strong **right skew** — many short stays and a long tail of a few very long ones.
Ordinary least squares assumes a symmetric, constant-variance, normally distributed error and can even
predict **negative** days. **Poisson regression** models the log of an expected count and guarantees
non-negative predictions, but assumes mean = variance. LOS is almost always **over-dispersed**
(variance ≫ mean), which is exactly what **Negative Binomial** regression handles by adding a
dispersion parameter — making it the most appropriate of the four for skewed LOS data. The tree models
(RF, XGBoost) make no distributional assumption and provide a non-parametric benchmark.

Reported per model: **MAE**, **RMSE**, **R²**, a model-comparison chart, and a predicted-vs-actual
scatter for the best model.

### 3A · Hospital LOS — by category

In [ ]:
COH3H = df_main[df_main["hospital_los_days"] > 0].copy()
y3h   = COH3H["hospital_los_days"].astype(float)
INC3  = ["n_incident_cats", "age", "gender", "insurance", "had_icu"]
X3ac  = make_X(COH3H, "category", INC3)

specs = {
    "Negative Binomial": GLMCount("negbin"),
    "Poisson":           GLMCount("poisson"),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
}
res3a = run_regression(X3ac, y3h, specs, "Exp 3A — Hospital LOS (categories)")

### 3B · Hospital LOS — by subcategory

In [ ]:
X3bs = make_X(COH3H, "subcategory", INC3)
specs = {
    "Negative Binomial": GLMCount("negbin"),
    "Poisson":           GLMCount("poisson"),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
}
res3b = run_regression(X3bs, y3h, specs, "Exp 3B — Hospital LOS (subcategories)")

### 3C · ICU LOS — by category

In [ ]:
COH3I = df_main[(df_main["had_icu"]) & (df_main["icu_los_days"] > 0)].copy()
y3i   = COH3I["icu_los_days"].astype(float)
X3cc  = make_X(COH3I, "category", INC3)

specs = {
    "Negative Binomial": GLMCount("negbin"),
    "Poisson":           GLMCount("poisson"),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
}
res3c = run_regression(X3cc, y3i, specs, "Exp 3C — ICU LOS (categories)")

### 3D · ICU LOS — by subcategory

In [ ]:
X3ds = make_X(COH3I, "subcategory", INC3)
specs = {
    "Negative Binomial": GLMCount("negbin"),
    "Poisson":           GLMCount("poisson"),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
}
res3d = run_regression(X3ds, y3i, specs, "Exp 3D — ICU LOS (subcategories)")

**Interpretation.** LOS is hard to predict from incident type + basic demographics alone, so
expect modest R². Watch whether Negative Binomial beats Poisson (evidence of over-dispersion) and
whether the tree models edge out the count GLMs. The predicted-vs-actual scatter should hug the
diagonal at short stays and fan out (under-predict) in the long tail — the classic signature of skewed
LOS data.

## Experiment 4 · Demographic-Based Prediction (age + gender + incident → outcomes)

**Purpose — can demographics alone predict outcomes, or do you need clinical features?** Here the
feature set is deliberately stripped down to **age + gender + incident category/subcategory** (no LOS,
no ICU flag, no GCS). Comparing these results against Experiments 1–3 shows how much predictive power
comes from demographics + mechanism versus the richer clinical features.

Six sub-experiments: Hospital LOS (4A/4B), ICU LOS (4C/4D), Discharge destination (4E/4F).
**Models:** Linear/Logistic Regression, Random Forest, XGBoost, CatBoost.

### 4A · Hospital LOS — by category

In [ ]:
INC4 = ["age", "gender"]
X4a  = make_X(COH3H, "category", INC4)
specs = {
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
    "CatBoost":          CatBoostRegressor(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4a = run_regression(X4a, y3h, specs, "Exp 4A — Hospital LOS, demographics (categories)")

### 4B · Hospital LOS — by subcategory

In [ ]:
X4b = make_X(COH3H, "subcategory", INC4)
specs = {
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
    "CatBoost":          CatBoostRegressor(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4b = run_regression(X4b, y3h, specs, "Exp 4B — Hospital LOS, demographics (subcategories)")

### 4C · ICU LOS — by category

In [ ]:
X4c = make_X(COH3I, "category", INC4)
specs = {
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
    "CatBoost":          CatBoostRegressor(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4c = run_regression(X4c, y3i, specs, "Exp 4C — ICU LOS, demographics (categories)")

### 4D · ICU LOS — by subcategory

In [ ]:
X4d = make_X(COH3I, "subcategory", INC4)
specs = {
    "Linear Regression": LinearRegression(),
    "Random Forest":     RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":           XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE),
    "CatBoost":          CatBoostRegressor(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4d = run_regression(X4d, y3i, specs, "Exp 4D — ICU LOS, demographics (subcategories)")

### 4E · Discharge destination — by category

In [ ]:
X4e = make_X(COH2, "category", INC4)
specs = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "Random Forest":       RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.1,
                                         eval_metric="mlogloss", random_state=RANDOM_STATE),
    "CatBoost":            CatBoostClassifier(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4e = run_classification(X4e, y2, specs, "Exp 4E — discharge, demographics (categories)")
comparison_chart(res4e, ["Accuracy", "F1_weighted"], "Exp 4E — model comparison")

### 4F · Discharge destination — by subcategory

In [ ]:
X4f = make_X(COH2, "subcategory", INC4)
specs = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    "Random Forest":       RandomForestClassifier(n_estimators=400, random_state=RANDOM_STATE, n_jobs=-1),
    "XGBoost":             XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.1,
                                         eval_metric="mlogloss", random_state=RANDOM_STATE),
    "CatBoost":            CatBoostClassifier(iterations=400, random_state=RANDOM_STATE, verbose=0),
}
res4f = run_classification(X4f, y2, specs, "Exp 4F — discharge, demographics (subcategories)")
comparison_chart(res4f, ["Accuracy", "F1_weighted"], "Exp 4F — model comparison")

**Conclusion — demographics vs. clinical features.** Compare Experiment 4 against the matching
clinical-feature runs:
- **4A–4D vs. Experiment 3:** if R² collapses once `had_icu`/`n_incident_cats` are removed, LOS is
  driven by clinical course, not demographics + mechanism.
- **4E–4F vs. Experiment 2:** discharge destination leans heavily on `hospital_los_days`/`had_icu`, so
  the demographics-only models should lose most of their edge on the non-Home classes (lower macro F1).

The takeaway is typically that **demographics + incident type set a weak baseline; clinical features
(LOS, ICU involvement, severity) carry the real predictive signal.**